# 5-2: Text Classification in Python

In this tutorial, we'll learn how to build a simple text classifier with Python. Our example uses music-review data from the course `data` folder. The task is to predict whether a review has a high score based on the language of the review.

The goal is to understand the basic supervised machine-learning workflow for text:

1. Define a classification task
2. Create labels
3. Split data into training and test sets
4. Turn text into numeric features
5. Train a model
6. Evaluate the model
7. Interpret what the model learned
8. Compare a few common machine-learning methods

By the end, you should be able to explain what features, labels, train/test splits, baselines, pipelines, and cross-validation mean in a text-classification project.


## Learning Objectives

1. Explain the difference between supervised and unsupervised learning
2. Define a text-classification task using local course data
3. Convert review text into TF-IDF features
4. Train a logistic-regression text classifier with `scikit-learn`
5. Evaluate a classifier with accuracy, a classification report, and a confusion matrix
6. Use cross-validation to estimate model performance
7. Compare logistic regression with Naive Bayes and a linear support vector machine


## Classification: An Overview

__Classification__ is a supervised machine-learning task. "Supervised" means that the model learns from examples that already have labels. It's a relatively straightforward definition of "learning": the model looks at examples with certain classification labels (positive reviews vs. negative reviews), identifies and learns patterns in the text of those reviews, then uses those learned patterns to classify other examples.

Text classification, in other words, can work with all sorts of texts and labels. Here's some examples:

| text | label |
| --- | --- |
| A movie review | positive or negative |
| An email | spam or not spam |
| A newspaper article | politics, sports, arts, etc. |
| A music review | high score or lower score |
| A text of fiction | scifi, western, romance, literary, other genres, etc. |

In any classication task, the model doesn't understand or "learn" about the text in a human way. Rather, it learns patterns between words and labels. If certain words or phrases often appear in high-scoring reviews, for example, the model can learn to associate those words with the high-score class.

### Important Interpretive Caution

A classifier is only as meaningful as the labels and data used to train it.

In this notebook, we'll create a binary label from review scores: reviews with scores of 75 or higher will count as high-scoring reviews. That threshold is a modeling choice, not a natural fact about music criticism.

As digital humanists, we should keep asking:

- Who created the labels?
- What do the labels leave out?
- What patterns might the model learn from genre, publication style, or reviewer habits?
- Does the model help us ask better questions, or does it flatten the evidence too much?

Classification can be useful, but it should not replace interpretation. Quite often, machine-learning classification in DH is actually used to just narrow examples for further interpretation and analysis, or help to identify texts that might otherwise get overlooked in analysis. It's a helpful filtering step, basically, but it may not tell you much unless you follow up with closer analyses of the texts being classified.

## The Music Review Dataset

We'll use `../data/music_reviews_clean.csv`. Each row is a music review. The dataset includes metadata such as album, artist, genre, critic, and score, along with a `body` column containing the review text.

Our task will be:

> Given the text of a music review, can we predict whether the review has a score of 75 or higher?

This is a __binary classification__ task because there are two possible outcomes: high score or lower score. Generally speaking, binary classification tends to be easier to implement and often more accurate than multi-classification (more than two classification labels/categories). Consider our current example. Would you have an easier time identifying a music review as positive versus negative? Or, would it be easier to identify reviews as either one, two, three, four, or five stars? With more possibilities, it becomes harder, for sure. The same is true for machine learning.

This is another important part of text classification. It's important to consider how distinct your categories are. Sometimes, text classification will reveal that your categories are actually fuzzy in the given corpora. Consider trying to parse the difference in literary genre. Can you perfectly differentiate between detective fiction versus mystery/thriller? How about music reviews that are impartial? The 3.5/5-star cases? That's not to say you couldn't use text classification to study these fuzzy categories, but it can be harder than binary classification.

## Setup

We'll use `pandas` for data, `matplotlib` and `seaborn` for plots, and `scikit-learn` for machine learning.

The main new pieces from `scikit-learn` are:

- `TfidfVectorizer` to turn text into numeric features
- `train_test_split` to create training and test sets
- `Pipeline` to combine vectorization and modeling
- `LogisticRegression` for our main classifier
- evaluation tools such as `classification_report` and `ConfusionMatrixDisplay`


In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
)
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC


## Step 1: Load The Data

Let's read the local CSV file. We only need a few columns for this lesson, but keeping the metadata visible helps us remember that these reviews have authors, genres, albums, and publication contexts.


In [ ]:
reviews = pd.read_csv("../data/music_reviews_clean.csv")

reviews.shape

Let's preview the table. The `body` column is the text we will classify. The `score` column is the numeric rating that we'll use to create our label.


In [ ]:
reviews.head()

Here is one review body. Notice that these are short review snippets rather than long essays. That matters because short documents often give a classifier less evidence to work with.


In [ ]:
reviews.loc[0, "body"]

## Step 2: Create A Label

Machine-learning models need labels. Our dataset has numeric scores, but for this lesson we want a simple two-class task.

We'll create a new column called `is_high_score`:

- `1` means the review score is 75 or higher
- `0` means the review score is below 75

This threshold gives us a reasonably balanced classification task. It's also easy to implement with conditionals, like this:

In [ ]:
reviews["is_high_score"] = (reviews["score"] >= 75).astype(int)
reviews["score_label"] = reviews["is_high_score"].map({
    0: "lower_score",
    1: "high_score",
})

reviews[["album", "artist", "score", "score_label", "body"]].head()

Let's look at the label counts. A classifier that sees a very imbalanced dataset can look deceptively good by always predicting the majority class. Here the classes are not perfectly balanced, but not too bad. Looks like we have over 2,000 examples per class to train our model: 

In [ ]:
reviews["score_label"].value_counts()

A quick plot makes the class balance easier to see.


In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(
    data=reviews,
    x="score_label",
    order=["lower_score", "high_score"],
    color="steelblue",
)
plt.title("Review Labels")
plt.xlabel("Label")
plt.ylabel("Number of reviews")
plt.tight_layout()

The score distribution also helps explain the threshold. The vertical line marks the cutoff we chose for `high_score`.


In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(
    reviews["score"],
    bins=25,
    color="steelblue",
    edgecolor="white",
)
plt.axvline(75, color="darkred", linestyle="--", label="High-score cutoff")
plt.title("Distribution of Review Scores")
plt.xlabel("Score")
plt.ylabel("Number of reviews")
plt.legend()
plt.tight_layout()


Does this distribution make you think about how the model will perform? How will it do on scores close to the threshold?

## Step 3: Define `X` And `y`

In machine-learning notation, `X` usually means the input features, and `y` usually means the label we want to predict.

For this project:

- `X` is the review text in the `body` column
- `y` is the binary label in the `is_high_score` column

The model will learn from the relationship between these two columns.


In [ ]:
X = reviews["body"]
y = reviews["is_high_score"]

print(f"Number of texts: {len(X)}")
print(f"Number of labels: {len(y)}")

## Step 4: Create Training And Test Sets

We need to split the data before training. This is a common part of machine learning, splitting the data into:

- The __training set__ to teach the model
- The __test set__ to evaluate the model on examples it did not see during training

This matters because a model can memorize training data. We want to know whether it can generalize to new reviews.

We'll use `stratify=y` so that the train and test sets keep roughly the same balance of high-score and lower-score reviews.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

print(f"Training documents: {len(X_train)}")
print(f"Testing documents: {len(X_test)}")


## Step 5: Establish A Baseline

Before training a real classifier, we should set a baseline. A baseline is a simple point of comparison. It answers the question: how well could we do with a very simple rule?

Here, the baseline model always predicts the most common class in the training data. If our real model cannot beat this baseline, then it has not learned anything useful from the review text.

We'll use `DummyClassifier`, which comes from `sklearn.dummy` in the `scikit-learn` library. A dummy classifier is intentionally simple. It does not look for meaningful patterns in the words. With `strategy="most_frequent"`, it checks which label appears most often in `y_train`, then predicts that label for every review in `X_test`.

That makes it a useful reality check. If the most common label is already 54 percent of the data, then a model that scores 55 percent is not very impressive.


In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

baseline_predictions = baseline.predict(X_test)
baseline_accuracy = accuracy_score(y_test, baseline_predictions)

print(f"Baseline accuracy: {baseline_accuracy:.3f}")


What does the baseline accuracy tell us? It's about 54%. What does that mean?

It means that a model can get about 54% of the test examples correct without reading the reviews at all. It simply guesses the most common label every time.

So when we train real classifiers, we should ask whether they beat this baseline by a meaningful amount. A model that improves only slightly over the baseline may technically be learning something, but it may not be useful enough for interpretation or analysis.


## Step 6: Turn Text Into TF-IDF Features

Machine-learning models do not take raw text as input. They need numbers.

A common way to represent text is TF-IDF, short for "term frequency-inverse document frequency." TF-IDF gives higher weight to terms that are important in a document but not common across every document.

The basic idea has two parts:

- __Term frequency__: a word or phrase gets more weight when it appears in a document.
- __Inverse document frequency__: a word or phrase gets less weight when it appears across many documents.

This is useful for text classification because it turns each review into a row of numeric features. Each column represents a word or phrase from the vocabulary. The number in each cell says how important that word or phrase is in that review, relative to the rest of the corpus.

TF-IDF is helpful because it reduces the influence of extremely common words and gives the classifier more useful evidence. For example, a word that appears in almost every review is probably less helpful for distinguishing high-scoring from lower-scoring reviews. A phrase that appears often in one kind of review but rarely elsewhere may be much more informative.

We'll use `TfidfVectorizer` with a few settings:

- `stop_words="english"` removes common English words
- `min_df=5` keeps only terms that appear in at least five documents
- `max_features=5000` keeps the vocabulary manageable
- `ngram_range=(1, 2)` includes single words and two-word phrases

Notice that we use `fit_transform()` on the training data, then `transform()` on the test data. That means the vocabulary and IDF weights are learned from the training set only, and the test set is converted using those same columns.


In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    min_df=5,
    max_features=5000,
    ngram_range=(1, 2),
)

tfidf_train = tfidf_vectorizer.fit_transform(X_train)
tfidf_test = tfidf_vectorizer.transform(X_test)

print(type(tfidf_train))
print(tfidf_train.shape)

This matrix has one row per training review and one column per vocabulary term. Like many text matrices, it's stored as a sparse matrix because most reviews contain only a small fraction of the vocabulary.

Like the other matrices, it's also not really in a form that's easy for humans to read. But we can take a look and preview some learned feature names with the `get_feature_names_out()` method:

In [ ]:
feature_names = tfidf_vectorizer.get_feature_names_out()
feature_names[100:150]

And if we wanted to review these features (i.e. tokens) with their corresponding TF-IDF scores, we could convert the data into a dataframe, like this:

In [ ]:
# just first five docs/rows and top 10 terms

for doc_id in range(5):
    row = tfidf_train[doc_id]

    nonzero = row.nonzero()[1]
    values = row.data

    doc_tfidf = pd.DataFrame({
        "term": feature_names[nonzero],
        "tfidf": values,
    }).sort_values("tfidf", ascending=False)

    print(f"\nDocument {doc_id}")
    print(doc_tfidf.head(10))

With these weights and terms, we have something to work with. TF-IDF to the rescue!

## Step 7: Train A Logistic Regression Classifier

Now we'll train our main model: __logistic regression.__

Despite the word "regression," logistic regression is commonly used for classification. For our binary task, it learns a weight for each word or phrase. Positive weights push the model toward the `high_score` class. Negative weights push it toward the `lower_score` class.

In simplified terms, the model adds up the evidence from all the TF-IDF features in a review. If words and phrases with positive weights are strong enough, the model predicts `high_score`. If words and phrases with negative weights are stronger, it predicts `lower_score`.

Logistic regression can also turn that evidence into a probability between 0 and 1. By default, a probability above 0.50 becomes class `1`, or `high_score`, while a probability below 0.50 becomes class `0`, or `lower_score`.

We'll use a `Pipeline` so the vectorizer and classifier stay connected. The pipeline first turns text into TF-IDF features, then passes those features to logistic regression.

The pipeline also prevents a common mistake called __data leakage__. Data leakage happens when information from the test set accidentally influences the training process. For example, if we fit the TF-IDF vectorizer on all reviews before splitting or before cross-validation, then the vocabulary and IDF scores would already contain information from the examples we are pretending are unseen. That can make evaluation scores look better than they really are.

By putting `TfidfVectorizer` inside the pipeline, each training split learns its vocabulary and IDF weights only from the training data available in that split.


In [ ]:
logit_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        stop_words="english",
        min_df=5,
        max_features=5000,
        ngram_range=(1, 2),
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42,
    )),
])

logit_model.fit(X_train, y_train)


Now let's deploy the model to predict labels for the test set. These are reviews the model did not see during training.

In [ ]:
test_predictions = logit_model.predict(X_test)
test_accuracy = accuracy_score(y_test, test_predictions)

print(f"Logistic regression accuracy: {test_accuracy:.3f}")


What does this accuracy score tell us? How does it compare to the baseline?

Accuracy tells us the overall share of correct predictions. It's important, but it's not the whole story.

Accuracy should also be contextualized alongside these metrics:

- __precision__: when the model predicts this class, how often is it right?
- __recall__: of the examples that truly belong to this class, how many did the model find?
- __f1-score__: a combined measure of precision and recall

We can review these metrics with the classification_report() function:

In [ ]:
print(classification_report(
    y_test,
    test_predictions,
    target_names=["lower_score", "high_score"],
))

Where is this logistic regression model scoring well? Where is it doing poorly?

A __confusion matrix__ is a table that compares the true labels to the labels predicted by the model. In this binary task, it lets us count four kinds of outcomes:

- lower-score reviews correctly predicted as `lower_score`
- lower-score reviews incorrectly predicted as `high_score`
- high-score reviews incorrectly predicted as `lower_score`
- high-score reviews correctly predicted as `high_score`

The diagonal cells are correct predictions. The off-diagonal cells are mistakes.

This is useful because two models can have similar accuracy but make different kinds of errors. Here, the confusion matrix helps us see whether the model is better at recognizing one class than the other.


In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_predictions,
    display_labels=["lower_score", "high_score"],
    cmap="Blues",
    colorbar=False,
)
plt.title("Logistic Regression Confusion Matrix")
plt.tight_layout()


### What If The Accuracy Is Only Modest?

Do not be surprised if this logistic regression classifier only improves a little over the baseline. That is useful information, not a failure of the process.

These reviews are short, and our label is created from a score threshold. A review with a score of 74 and a review with a score of 75 may not be very different in language, even though our binary label puts them in different classes. Remember: classification performance depends on the quality and clarity of the labels as much as it depends on the machine-learning method.

## Step 8: Inspect Prediction Probabilities

Logistic regression can also estimate probabilities. For example, the model might predict that a review has a 72 percent chance of being high-scoring.

We get those values with `.predict_proba()`. This method returns a table-like array with one row per review and one column per possible class. For this binary classifier, the columns correspond to class `0` and class `1`.

The code below uses `[:, 1]` to select the second column: the probability of class `1`, which is our `high_score` label.

These probabilities can help us inspect borderline cases. A prediction near 0.50 is less confident than one near 0.95. They can also help us see when a model technically makes a prediction but is not very sure about it.

The preview table combines several pieces of information:

- the review text from `X_test`
- the true label from `y_test`
- the predicted label from `test_predictions`
- the model's estimated probability that the review is `high_score`

This kind of table is often more useful for interpretation than a single accuracy score because it lets us inspect individual cases.


In [ ]:
high_score_probabilities = logit_model.predict_proba(X_test)[:, 1]

prediction_preview = pd.DataFrame({
    "review": X_test.iloc[:8].values,
    "actual_label": y_test.iloc[:8].map({0: "lower_score", 1: "high_score"}).values,
    "predicted_label": pd.Series(test_predictions[:8]).map({0: "lower_score", 1: "high_score"}),
    "probability_high_score": high_score_probabilities[:8],
})

prediction_preview


We can also write a few new mini-reviews and ask the model how it would classify them. This is not a rigorous evaluation; it is just a way to see how the trained classifier behaves.


In [ ]:
new_reviews = pd.Series([
    "A brilliant, generous album with sharp songwriting and gorgeous production.",
    "The record feels dull, unfocused, and short on memorable songs.",
    "It has a few strong moments, but the album never quite comes together.",
    "This is a terrible and wonderful album of outstandingly bad quality for those with no taste."
])

new_predictions = logit_model.predict(new_reviews)
new_probabilities = logit_model.predict_proba(new_reviews)[:, 1]

pd.DataFrame({
    "review": new_reviews,
    "predicted_label": pd.Series(new_predictions).map({0: "lower_score", 1: "high_score"}),
    "probability_high_score": new_probabilities,
})

## Step 9: Interpret Important Features

One advantage of logistic regression is that we can inspect the feature weights.

A positive weight means the term pushes the model toward predicting `high_score`. A negative weight means the term pushes the model toward predicting `lower_score`.

These weights do not prove that a word causes a high or low score. They only show which terms were useful to this model for separating the two classes.


In [ ]:
fitted_vectorizer = logit_model.named_steps["tfidf"]
fitted_classifier = logit_model.named_steps["classifier"]

feature_names = fitted_vectorizer.get_feature_names_out()
coefficients = fitted_classifier.coef_[0]

top_high_indices = np.argsort(coefficients)[-20:][::-1]
top_lower_indices = np.argsort(coefficients)[:20]

top_high_terms = pd.DataFrame({
    "term": feature_names[top_high_indices],
    "weight": coefficients[top_high_indices],
    "direction": "high_score",
})

top_lower_terms = pd.DataFrame({
    "term": feature_names[top_lower_indices],
    "weight": coefficients[top_lower_indices],
    "direction": "lower_score",
})

important_terms = pd.concat([top_high_terms, top_lower_terms], ignore_index=True)
important_terms

Let's plot those feature weights. The words on the right push toward high-score predictions. The words on the left push toward lower-score predictions.


In [ ]:
plot_terms = important_terms.copy()
plot_terms["term_direction"] = plot_terms["term"] + " (" + plot_terms["direction"] + ")"

plt.figure(figsize=(8, 9))
sns.barplot(
    data=plot_terms.sort_values("weight"),
    x="weight",
    y="term_direction",
    hue="direction",
    dodge=False,
)
plt.axvline(0, color="black", linewidth=1)
plt.title("Terms With Strong Logistic Regression Weights")
plt.xlabel("Model weight")
plt.ylabel("Term")
plt.legend(title="Direction")
plt.tight_layout()

## Step 10: Cross-Validation

A single train/test split is useful, but it can be a little dependent on which examples happened to land in the test set.

__Cross-validation__ gives us a more stable estimate. It splits the data several different ways, trains the model several times, and reports the average performance.

Here we'll use 5-fold cross-validation. This is easy to implement with `scikit-learn`:

In [ ]:
cv_results = cross_validate(
    logit_model,
    X,
    y,
    cv=5,
    scoring=["accuracy", "f1"],
)

print("Accuracy scores:", cv_results["test_accuracy"])
print("F1 scores:", cv_results["test_f1"])
print(f"Mean accuracy: {cv_results['test_accuracy'].mean():.3f}")
print(f"Mean F1: {cv_results['test_f1'].mean():.3f}")

## Step 11: A Few Other Classification Methods

Logistic regression is a strong, interpretable starting point, but it is not the only option. And as we've seen here, it's not always going to have the best classification results. So let's compare other methods. Consider these:

- __Dummy baseline__: always predicts the most common class. It gives us a simple minimum comparison point.
- __Logistic regression__: learns weighted evidence for each class. It is often a good choice when we want both reasonable performance and interpretable feature weights.
- __Naive Bayes__: a fast classic method for text classification. It estimates which words are more likely to appear in each class, then uses those estimates to classify new documents. It is called "naive" because it makes a simplifying assumption that features are independent from one another. That assumption is not really true for language, but the method can still work surprisingly well for text.
- __Linear SVM__: a support vector machine with a linear decision boundary. It tries to find a boundary that separates the classes with as much margin as possible. Linear SVMs often perform well on TF-IDF text features, but `LinearSVC` does not provide probabilities by default.

There are many other possibilities, including random forests, k-nearest neighbors, neural networks, and transformer models. Those can be useful, but for many beginner text-classification projects, TF-IDF plus logistic regression, Naive Bayes, or a linear SVM is a very reasonable place to start.

We'll keep the same TF-IDF settings for the real classifiers so the comparison stays simple. Each of these classification methods is simple to implement with `scikit-learn`.

The code below sets up the comparison in two parts:

- `tfidf_settings` stores the vectorizer settings in one dictionary so we do not repeat the same arguments several times.
- `models` is a dictionary where each key is a model name and each value is a `Pipeline` that combines TF-IDF vectorization with a classifier.

This structure lets us loop through the models and evaluate each one in the same way.


In [ ]:
tfidf_settings = {
    "stop_words": "english",
    "min_df": 5,
    "max_features": 5000,
    "ngram_range": (1, 2),
}

models = {
    "Dummy baseline": Pipeline([
        ("tfidf", TfidfVectorizer(**tfidf_settings)),
        ("classifier", DummyClassifier(strategy="most_frequent")),
    ]),
    "Naive Bayes": Pipeline([
        ("tfidf", TfidfVectorizer(**tfidf_settings)),
        ("classifier", MultinomialNB()),
    ]),
    "Logistic Regression": Pipeline([
        ("tfidf", TfidfVectorizer(**tfidf_settings)),
        ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
    ]),
    "Linear SVM": Pipeline([
        ("tfidf", TfidfVectorizer(**tfidf_settings)),
        ("classifier", LinearSVC(random_state=42, max_iter=3000)),
    ]),
}


Now we'll cross-validate each model. This may take a little longer than fitting one classifier because each model is trained several times.

Here is what the code is doing:

- `model_rows = []` creates an empty list where we will store the results.
- `for model_name, model in models.items():` loops through each named pipeline in the `models` dictionary.
- `cross_validate()` trains and evaluates the model using 5-fold cross-validation.
- `cv=5` means the data is split into five parts. Each part gets one turn as the test fold while the other four parts are used for training.
- `scoring=["accuracy", "f1"]` asks for two evaluation metrics.
- `scores["test_accuracy"].mean()` calculates the average accuracy across the five folds.
- `scores["test_f1"].mean()` calculates the average F1 score across the five folds.
- `accuracy_std` tells us how much the accuracy varied across the folds.

Finally, we turn the list of result dictionaries into a DataFrame and sort the models by mean accuracy.


In [ ]:
model_rows = []

for model_name, model in models.items():
    scores = cross_validate(
        model,
        X,
        y,
        cv=5,
        scoring=["accuracy", "f1"],
    )

    model_rows.append({
        "model": model_name,
        "mean_accuracy": scores["test_accuracy"].mean(),
        "mean_f1": scores["test_f1"].mean(),
        "accuracy_std": scores["test_accuracy"].std(),
    })

model_results = pd.DataFrame(model_rows).sort_values("mean_accuracy", ascending=False)
model_results

A small plot makes the model comparison easier to read. The baseline matters: the real classifiers should beat it by a meaningful amount.


In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(
    data=model_results.sort_values("mean_accuracy"),
    x="mean_accuracy",
    y="model",
    color="steelblue",
)
plt.xlim(0.45, 0.85)
plt.title("Cross-Validated Accuracy by Model")
plt.xlabel("Mean accuracy")
plt.ylabel("Model")
plt.tight_layout()

## Reflection: How Could the Model Improve?

The mean accuracy and F1 scores across these models are not especially strong. That does not mean the code failed. It may mean the classification task is genuinely difficult. How could this be improved?

We could consider using more data, improving the labels, trying different thresholds, adding metadata features, tuning model settings, or using a more powerful model. But the most important issue may be the label definition itself.

Right now, reviews just below and just above the cutoff are treated as different classes. A review with a score of 74 is `lower_score`, while a review with a score of 75 is `high_score`. Those two reviews may not sound very different.

One experiment would be to remove the middle range and compare only clearer cases:

- `high_score`: scores greater than 90
- `lower_score`: scores less than 70

This asks a slightly different question: can the models distinguish strongly positive reviews from clearly lower-scoring reviews? We won't touch on this possible next step here, but it points to the nature of machine learning classification––that it's an iterative process, one where questions get raised along the way and testing models leads to more refined questions and/or more refined models.